# 02 - SQL analysis in SQLite

We load the two clean files into a small SQL database and answer the project questions with queries. SQL is a core analyst skill, so each query practises a specific technique.

**Two tables:**
- `states`: one row per state with all the metrics from notebook 01
- `installs`: one row per state and year with the number of stations installed

| Query | Question | SQL idea |
|---|---|---|
| 1 | Which states have the most EVs per charging station? | `ORDER BY`, `LIMIT` |
| 2 | Which states combine high adoption with high charging pressure? | window functions (`RANK`) |
| 3 | Which states are "priority gap" states? | CTE with national benchmarks |
| 4 | Where did the later build-out fall short of demand? | filtering on a calculated gap |
| 5 | How fast is national installation growing? | `LAG`, running total |
| 6 | Which states have the lowest share of working stations? | `HAVING` |
| 7 | Which states accelerated or slowed in FY2024-25? | pivot with `CASE WHEN` |
| 8 | How concentrated are EVs and stations in the top 5 states? | subquery |

## Step 1 - Create the database

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd

states = pd.read_csv("../data/clean/ev_state_metrics.csv")
installs = pd.read_csv("../data/clean/charger_installs_by_year.csv")

conn = sqlite3.connect("../data/clean/ev_gap.db")
states.to_sql("states", conn, if_exists="replace", index=False)
installs.to_sql("installs", conn, if_exists="replace", index=False)
conn.commit()

def run(sql):
    """Run a SQL query and return the result as a table."""
    return pd.read_sql_query(sql, conn)

ALL_QUERIES = []

run("SELECT 'states' AS table_name, COUNT(*) AS rows_count FROM states UNION ALL SELECT 'installs', COUNT(*) FROM installs")

## Query 1 - States with the most EVs per charging station
Only states with at least 20,000 EVs (`in_analysis_set = 1`), so that tiny states do not distort the ranking.

In [ ]:
q = """
SELECT state,
       ev_total,
       chargers_to_fy24,
       ROUND(evs_per_charger_fy24, 0) AS evs_per_charger
FROM states
WHERE in_analysis_set = 1
ORDER BY evs_per_charger_fy24 DESC
LIMIT 10
"""
ALL_QUERIES.append(("Query 1 - States with the most EVs per charging station", q))
run(q)

## Query 2 - Adoption rank and pressure rank together
`RANK() OVER (ORDER BY ...)` gives each state a position without grouping the rows. We add the two ranks, so a low combined score means "high adoption and high charging pressure".

In [ ]:
q = """
WITH base AS (
    SELECT state, ev_penetration_pct, evs_per_charger_fy24
    FROM states
    WHERE in_analysis_set = 1
),
ranked AS (
    SELECT state,
           ev_penetration_pct,
           evs_per_charger_fy24,
           RANK() OVER (ORDER BY ev_penetration_pct DESC) AS adoption_rank,
           RANK() OVER (ORDER BY evs_per_charger_fy24 DESC) AS pressure_rank
    FROM base
)
SELECT state,
       ROUND(ev_penetration_pct, 2) AS ev_penetration_pct,
       ROUND(evs_per_charger_fy24, 0) AS evs_per_charger,
       adoption_rank,
       pressure_rank,
       adoption_rank + pressure_rank AS combined_score
FROM ranked
ORDER BY combined_score, state
LIMIT 10
"""
ALL_QUERIES.append(("Query 2 - Combined adoption and charging-pressure ranking", q))
run(q)

## Query 3 - Priority-gap states with national benchmarks
A **CTE** (`WITH ... AS`) calculates the national figures once, and we compare every state against them. This repeats the classification from notebook 01 in pure SQL.

In [ ]:
q = """
WITH nat AS (
    SELECT 100.0 * SUM(ev_total) / SUM(vehicles_total) AS nat_penetration,
           1.0 * SUM(ev_total) / SUM(chargers_to_fy24) AS nat_ratio
    FROM states
)
SELECT s.state,
       ROUND(s.ev_penetration_pct, 2) AS ev_penetration_pct,
       ROUND(s.evs_per_charger_fy24, 0) AS evs_per_charger,
       ROUND(nat.nat_penetration, 2) AS national_penetration,
       ROUND(nat.nat_ratio, 0) AS national_evs_per_charger
FROM states s, nat
WHERE s.in_analysis_set = 1
  AND s.ev_penetration_pct >= nat.nat_penetration
  AND s.evs_per_charger_fy24 >= nat.nat_ratio
ORDER BY s.evs_per_charger_fy24 DESC
"""
ALL_QUERIES.append(("Query 3 - Priority-gap states (high adoption and thinly served)", q))
run(q)

## Query 4 - Where did the later build-out fall short of demand?
For stations installed after March 2024, we compare each state's share of the new stations with its share of EVs. A negative gap means the state received a smaller share of new stations than its share of EVs.

In [ ]:
q = """
SELECT state,
       ROUND(ev_share_pct, 1) AS ev_share_pct,
       ROUND(new_charger_share_pct, 1) AS new_charger_share_pct,
       ROUND(new_vs_ev_share_pp, 1) AS gap_points,
       new_chargers_after_fy24
FROM states
WHERE in_analysis_set = 1
  AND new_vs_ev_share_pp < 0
ORDER BY new_vs_ev_share_pp
LIMIT 10
"""
ALL_QUERIES.append(("Query 4 - States that received less than their share of new stations", q))
run(q)

## Query 5 - National installation growth
`LAG()` looks at the previous row, so we can calculate year-on-year growth. `SUM() OVER (ORDER BY ...)` gives a running total. The last year is partial (to 1 March 2026), so we leave its growth blank instead of comparing it with a full year.

In [ ]:
q = """
WITH yearly AS (
    SELECT fiscal_year, SUM(chargers_installed) AS installed
    FROM installs
    GROUP BY fiscal_year
)
SELECT fiscal_year,
       installed,
       CASE WHEN fiscal_year LIKE '%to 1 Mar%' THEN NULL
            ELSE ROUND(100.0 * (installed - LAG(installed) OVER (ORDER BY fiscal_year))
                       / LAG(installed) OVER (ORDER BY fiscal_year), 1)
       END AS growth_vs_previous_year_pct,
       SUM(installed) OVER (ORDER BY fiscal_year) AS cumulative_installed
FROM yearly
ORDER BY fiscal_year
"""
ALL_QUERIES.append(("Query 5 - Yearly installs, growth and running total", q))
run(q)

## Query 6 - States with the lowest share of working stations
Only states with at least 50 installed stations (`HAVING`-style filter), so a handful of stations does not distort the rate.

In [ ]:
q = """
SELECT state,
       total_installed_net AS installed,
       operational,
       ROUND(operational_rate_pct, 1) AS operational_rate_pct
FROM states
WHERE total_installed_net >= 50
ORDER BY operational_rate_pct
LIMIT 8
"""
ALL_QUERIES.append(("Query 6 - Lowest operational rate (states with 50+ stations)", q))
run(q)

## Query 7 - Which states sped up or slowed down in FY2024-25?
`CASE WHEN` pivots the years into columns. A second CTE is needed because SQL cannot reuse a column alias in the same `SELECT`.

In [ ]:
q = """
WITH pivoted AS (
    SELECT state,
           SUM(CASE WHEN fiscal_year = 'FY2023-24' THEN chargers_installed ELSE 0 END) AS fy23_24,
           SUM(CASE WHEN fiscal_year = 'FY2024-25' THEN chargers_installed ELSE 0 END) AS fy24_25
    FROM installs
    GROUP BY state
)
SELECT state,
       fy23_24,
       fy24_25,
       ROUND(100.0 * (fy24_25 - fy23_24) / fy23_24, 1) AS change_pct
FROM pivoted
WHERE fy23_24 >= 100
ORDER BY change_pct DESC
LIMIT 10
"""
ALL_QUERIES.append(("Query 7 - Change in installs, FY2023-24 to FY2024-25 (states with 100+ in FY2023-24)", q))
run(q)

## Query 8 - How concentrated are EVs and stations?
A subquery picks the five states with the most EVs. We then compare their share of all EVs with their share of all charging stations (to March 2024).

In [ ]:
q = """
SELECT ROUND(100.0 * SUM(CASE WHEN state IN (SELECT state FROM states ORDER BY ev_total DESC LIMIT 5)
                              THEN ev_total ELSE 0 END) / SUM(ev_total), 1) AS top5_share_of_evs_pct,
       ROUND(100.0 * SUM(CASE WHEN state IN (SELECT state FROM states ORDER BY ev_total DESC LIMIT 5)
                              THEN chargers_to_fy24 ELSE 0 END) / SUM(chargers_to_fy24), 1) AS top5_share_of_stations_pct
FROM states
"""
ALL_QUERIES.append(("Query 8 - Top 5 EV states: share of EVs vs share of charging stations", q))
run(q)

## Step 2 - Save your queries to a .sql file

In [ ]:
lines = ["-- India EV Adoption and Charging Gap Analysis: analysis queries (SQLite)", ""]
for title, query in ALL_QUERIES:
    lines.append("-- " + title)
    lines.append(query.strip() + ";")
    lines.append("")

Path("../sql/analysis_queries.sql").write_text("\n".join(lines), encoding="utf-8")
print("Saved", len(ALL_QUERIES), "queries to ../sql/analysis_queries.sql")
conn.close()

## Checkpoint
Send me the outputs of Queries 1, 3, 4, 5 and 8.